# Primeros pasos con `schelling_matricial`

Este notebook muestra el flujo básico de análisis:

1. Una simulación del modelo clásico de Schelling (dos tipos, modelo P).
2. La retícula inicial y la final.
3. La evolución de la segregación barrido a barrido.
4. Un barrido de la tolerancia `tau` con réplicas, analizado con pandas y seaborn.

La referencia completa de la API está en `docs/manual_usuario.md`.
Usa el kernel del entorno `.venv_schelling`.

In [ ]:
from pathlib import Path
import dataclasses

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns

from schelling_matricial import Config, Simulacion, ejecutar_replicas

# Raíz del repositorio (la carpeta que contiene pyproject.toml), para guardar
# los resultados en <raíz>/resultados/, que está en .gitignore.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RESULTADOS = RAIZ / "resultados"

# Colores de la retícula: 0 = vacío (gris claro), 1 = tipo 1, 2 = tipo 2.
COLORES = ListedColormap(["#e8e8e8", "#1f77b4", "#ff7f0e"])

## 1. Una simulación

Modelo clásico: un vecino distinto molesta 1 y uno igual, 0. Con `tau=0.5`, un
agente está incómodo si más de la mitad de sus vecinos ocupados son del otro tipo.
`cada=1` guarda los observables en cada barrido para ver la evolución.

In [ ]:
cfg = Config(
    L=40,
    poblaciones=(700, 700),   # 1400 agentes; 200 casillas vacías
    H=[[0, 1], [1, 0]],
    tau=0.5,
    modelo="P",
    seed=123,
    cada=1,
)

sim = Simulacion(cfg)
reticula_inicial = sim.reticula.copy()   # copy(): la retícula cambia en su sitio
resultado = sim.ejecutar()

print("Motivo de parada:", resultado["motivo_parada"], "/", resultado["tipo_punto_fijo"])
print("Barridos:", resultado["barridos"], "| movimientos:", resultado["movimientos_totales"])
print(f"Segregación s = {resultado['s']:.3f} | homofilia h = {resultado['h']:.3f}")

## 2. Retícula inicial y final

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 5))
for eje, reticula, titulo in zip(
    ejes,
    [reticula_inicial, sim.reticula],
    ["Inicial (al azar)", f"Final (barrido {resultado['barridos']})"],
):
    eje.imshow(reticula, cmap=COLORES, vmin=0, vmax=2, interpolation="nearest")
    eje.set_title(titulo)
    eje.axis("off")
plt.show()

## 3. Evolución de la segregación

`resultado["serie"]` tiene los observables de cada barrido. La convertimos en
un `DataFrame` para graficarla.

In [ ]:
serie = pd.DataFrame(resultado["serie"])
serie["incomodos"] = serie["u"].apply(np.mean)   # fracción media de incómodos

fig, (eje_s, eje_u) = plt.subplots(1, 2, figsize=(11, 4))
eje_s.plot(serie["barrido"], serie["s"], marker="o")
eje_s.set(xlabel="Barrido", ylabel="Índice de segregación s", title="Segregación")
eje_u.plot(serie["barrido"], serie["incomodos"], marker="o", color="tab:red")
eje_u.set(xlabel="Barrido", ylabel="Fracción de incómodos", title="Incomodidad")
plt.tight_layout()
plt.show()

## 4. Barrido de la tolerancia con réplicas

Para cada valor de `tau` corremos varias réplicas (una semilla distinta cada
una). `ejecutar_replicas` devuelve una fila por réplica y la guarda en CSV.
Sube `semillas` para tener resultados más estables (tarda más).

In [ ]:
cfg_base = dataclasses.replace(cfg, cada=None)   # sin serie temporal: más rápido
valores_tau = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
semillas = range(5)

filas = []
for tau in valores_tau:
    filas += ejecutar_replicas(dataclasses.replace(cfg_base, tau=tau), semillas=semillas)

df = pd.DataFrame(filas)
RESULTADOS.mkdir(exist_ok=True)
df.to_csv(RESULTADOS / "barrido_tau.csv", index=False)
df[["tau_1", "seed", "s", "h", "barridos", "motivo_parada"]].head(10)

In [ ]:
resumen = df.groupby("tau_1")["s"].agg(["mean", "std"]).round(3)
print(resumen)

fig, eje = plt.subplots(figsize=(7, 4))
sns.pointplot(data=df, x="tau_1", y="s", errorbar="sd", ax=eje)
eje.set(xlabel="Tolerancia tau", ylabel="Índice de segregación s",
        title="Segregación final frente a tolerancia")
plt.show()

### Para seguir

- Cambia `modelo="S"` o usa tres tipos (ver las recetas del manual, §9).
- Guarda los barridos grandes en `resultados/` y analízalos en otro notebook.
- Usa `scikit-learn` para agrupar los estados finales o `SALib` para un
  análisis de sensibilidad de los parámetros.